In [1]:
import pandas as pd

#LOAD DATASETS
datafiles = {
    "Fenomenos": "../Dataset_INUMET/Fenomenos.csv",
    "Humedad": "../Dataset_INUMET/Humedad.csv",
    "Lluvia": "../Dataset_INUMET/Lluvia.csv",
    "Temperatura": "../Dataset_INUMET/Temperatura.csv",
    "Visibilidad": "../Dataset_INUMET/Visibilidad.csv",
    "VientoSalto": "../Dataset_INUMET/VientoSalto.csv",
    "VientoCarrasco": "../Dataset_INUMET/VientoCarrasco.csv"
}

dataframes = {}
for name, path in datafiles.items():
    dataframes[name] = pd.read_csv(path, sep = ";", encoding='utf-8')
# Access data like: dataframes["Temperature"]


In [2]:
# ====== 1. Lluvia.csv ======
print("Procesando datos Lluvia.csv")
df_lluvia = dataframes["Lluvia"].copy()

df_lluvia = df_lluvia.rename(columns={
    "Estación": "estacion",
    "Cod. Pluvio": "cod_pluvio",
    "Fecha": "fecha",
    "[mm]": "precip_mm",
    "Comentario": "coment_lluv"
})

# Convertir fecha y precipitación
df_lluvia["fecha"] = pd.to_datetime(df_lluvia["fecha"], format="%d/%m/%Y", errors="coerce")
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].replace("TRAZA", "0.0")  #! Nota: Transformación TRAZA se traduce a 0.0 
df_lluvia["precip_mm"] = df_lluvia["precip_mm"].astype(float)

#! Criterio: día lluvioso si precipitación >= 1.0 mm
precip_minima = 1.0
df_lluvia["dia_lluvioso"] = (df_lluvia["precip_mm"] >= precip_minima).astype(int)

# Seleccionamos solo columnas útiles para el df_global
df_lluvia = df_lluvia[["fecha", "precip_mm", "dia_lluvioso"]]

print(df_lluvia.head())


Procesando datos Lluvia.csv
       fecha  precip_mm  dia_lluvioso
0 1950-01-01        4.6             1
1 1950-01-02        0.0             0
2 1950-01-03        0.0             0
3 1950-01-04        0.0             0
4 1950-01-05        0.0             0


In [3]:
# ====== 2. Humedad.csv ======
print("Procesando datos Humedad.csv")
df_hum = dataframes["Humedad"].copy()

df_hum = df_hum.rename(columns={
    "Fecha/hora": "fecha",
    "HumedadCarrasco ": "hum_carrasco",  
    "HumedadSalto ": "hum_salto"
})

# Convertir fecha
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"], errors="coerce")
df_hum["fecha"] = df_hum["fecha"].dt.date  # Extraer solo la fecha (sin hora)
df_hum["fecha"] = pd.to_datetime(df_hum["fecha"])  # Volver a datetime

# Eliminar columnas no deseadas
df_hum = df_hum.drop(columns=["Unnamed: 3", "Unnamed: 4"], errors="ignore")  # errors="ignore" evita error si no existen

# Agrupar por fecha (en caso de múltiples mediciones por día)
df_hum = df_hum.groupby("fecha").agg({
    "hum_carrasco": "mean",
    "hum_salto": "mean"
}).reset_index()

print(df_hum.head())

Procesando datos Humedad.csv
       fecha  hum_carrasco  hum_salto
0 1980-01-01           NaN       69.0
1 1980-01-02          60.0       61.0
2 1980-01-03          76.0       63.0
3 1980-01-04          60.0       82.0
4 1980-01-05          66.0       84.0


In [31]:
# ====== 3. Temperatura.csv ======
print("Procesando datos Temperatura.csv")

df_temp = dataframes["Temperatura"].copy()

df_temp = df_temp.rename(columns={
    "Fecha": "fecha",
    "TmáxCarrasco": "tmax_carrasco",
    "TmínCarrasco": "tmin_carrasco",
    "TmáxSalto": "tmax_salto",
    "TmínSalto": "tmin_salto"
})

df_temp["fecha"] = pd.to_datetime(df_temp["fecha"], errors="coerce", dayfirst=True)
# Eliminar fechas no válidas
df_temp = df_temp.dropna(subset=["fecha"])
df_temp = df_temp.drop(columns=["Unnamed: 5"], errors="ignore")  # errors="ignore" evita error si no existen

#? Se añade datos media y amplitud térmica, por si se usan más adelante
# 1. Temperatura media
df_temp["tmedia_carrasco"] = (df_temp["tmin_carrasco"] + df_temp["tmax_carrasco"]) / 2
df_temp["tmedia_salto"] = (df_temp["tmin_salto"] + df_temp["tmax_salto"]) / 2

# 2. Amplitud térmica
df_temp["amplitud_carrasco"] = df_temp["tmax_carrasco"] - df_temp["tmin_carrasco"]
df_temp["amplitud_salto"] = df_temp["tmax_salto"] - df_temp["tmin_salto"]

# 3. Discretización en categorías climáticas
def categorizar_temp(t):
    if t < 2:
        return "gelido"
    elif 2 <= t < 12:
        return "frio"
    elif 12 <= t < 22:
        return "templado"
    elif 22 <= t < 30:
        return "calido"
    elif t >= 30:
        return "abrasador"
    else:
        return "ERROR"

#?Valores discretizados, clasificados según tmedia
df_temp["tm_carrasco"] = df_temp["tmedia_carrasco"].apply(categorizar_temp)
df_temp["tm_salto"] = df_temp["tmedia_salto"].apply(categorizar_temp)

df_temp = df_temp[["fecha", "tm_carrasco", "tm_salto"]]
print(df_temp.head(5))


Procesando datos Temperatura.csv
       fecha tm_carrasco   tm_salto
0 1970-01-01      calido     calido
1 1970-01-02      calido     calido
2 1970-01-03      calido     calido
3 1970-01-04    templado     calido
4 1970-01-05      calido  abrasador


In [35]:
# Merge de dataset procesados

print("Combinando datasets en df_global")

df_global = df_lluvia.copy()
# Hacer merge con humedad
df_global = pd.merge(df_global, df_hum, on="fecha", how="outer")

# Hacer merge con temperatura
#df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")
df_global = pd.merge(df_global, df_temp, on="fecha", how="outer")
print("df_global head:")
#print(df_global.head(10))
print(df_global[df_global["fecha"] >= "1980-01-01"].head(10))

Combinando datasets en df_global
df_global head:
       fecha  precip_mm  dia_lluvioso  hum_carrasco  hum_salto tm_carrasco  \
0 1950-01-01        4.6           1.0           NaN        NaN         NaN   
1 1950-01-02        0.0           0.0           NaN        NaN         NaN   
2 1950-01-03        0.0           0.0           NaN        NaN         NaN   
3 1950-01-04        0.0           0.0           NaN        NaN         NaN   
4 1950-01-05        0.0           0.0           NaN        NaN         NaN   
5 1950-01-06        0.0           0.0           NaN        NaN         NaN   
6 1950-01-07        0.0           0.0           NaN        NaN         NaN   
7 1950-01-08        0.0           0.0           NaN        NaN         NaN   
8 1950-01-09        0.0           0.0           NaN        NaN         NaN   
9 1950-01-10        0.0           0.0           NaN        NaN         NaN   

  tm_salto  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  
5      NaN  
6